# Re-verify this proof

1. Click into the code cell below and press **Shift+Enter** to run it.
2. A DOI input appears (already filled in from the link you clicked, and validated as you type). Click **Run proof**.
3. The proof fetches from Zenodo and runs — this takes 10–60 seconds. Output appears below the widget, ending with a line like `VERDICT: PROVED`.

To verify a different proof, paste its Zenodo DOI (`10.5281/zenodo.XXXXXXX`) into the input and click **Run proof** again.

<sub>Why a button instead of "Run All": anywidget syncs the DOI from browser to Python kernel asynchronously, so a Run All would race the fetch against the sync. The button is the sync barrier.</sub>

In [ ]:
import os
import re
import sys

import anywidget
import ipywidgets as widgets
import requests
import traitlets
from IPython.display import display

DOI_RE = re.compile(r"^10\.\d{4,9}/zenodo\.\d+$")
DOI_PRELOAD_FILE = "/tmp/binder_doi"


def _preloaded_doi() -> str:
    """Return a valid Zenodo DOI from the server-side capture file, or ''.

    The binder_doi_capture Jupyter Server extension intercepts ?doi= on the
    initial HTTP request (which JupyterLab's SPA router strips before any
    client-side JS can read it) and writes it here. If the file is absent
    or malformed, we just start with an empty widget.
    """
    try:
        with open(DOI_PRELOAD_FILE) as fh:
            value = fh.read().strip()
        if DOI_RE.fullmatch(value):
            return value
    except OSError:
        pass
    return ""


class DOIReader(anywidget.AnyWidget):
    _esm = """
    function render({ model, el }) {
      const DOI_RE = /^10\\.\\d{4,9}\\/zenodo\\.\\d+$/;
      const wrap = document.createElement('div');
      wrap.style.fontFamily = 'monospace';
      const label = document.createElement('label');
      label.textContent = 'Zenodo DOI: ';
      const input = document.createElement('input');
      input.value = model.get('doi') || '';
      input.placeholder = '10.5281/zenodo.XXXXXXX';
      input.size = 40;
      const status = document.createElement('span');
      status.style.marginLeft = '0.5em';
      const btn = document.createElement('button');
      btn.textContent = 'Run proof';
      btn.style.marginLeft = '0.5em';
      btn.disabled = true;
      function update() {
        const v = input.value.trim();
        if (!v) { status.textContent = ''; btn.disabled = true; return; }
        if (DOI_RE.test(v)) {
          status.textContent = '✓ valid';
          status.style.color = 'green';
          btn.disabled = false;
          model.set('doi', v);
          model.save_changes();
        } else {
          status.textContent = '✗ not a Zenodo DOI';
          status.style.color = 'red';
          btn.disabled = true;
        }
      }
      btn.addEventListener('click', () => {
        // Bump the trigger traitlet. The Python-side observer listens
        // for changes and does the fetch/exec.
        model.set('trigger', (model.get('trigger') || 0) + 1);
        model.save_changes();
      });
      input.addEventListener('input', update);
      input.addEventListener('change', update);
      update();
      label.appendChild(input);
      wrap.appendChild(label);
      wrap.appendChild(status);
      wrap.appendChild(btn);
      el.appendChild(wrap);
    }
    export default { render };
    """
    doi = traitlets.Unicode("").tag(sync=True)
    trigger = traitlets.Int(0).tag(sync=True)


output = widgets.Output()


def run_proof(doi: str) -> None:
    if not DOI_RE.fullmatch(doi):
        raise ValueError(
            f"Invalid Zenodo DOI: {doi!r}. Expected 10.5281/zenodo.XXXXXXX."
        )
    zenodo_id = doi.split("/zenodo.")[-1]
    print(f"Fetching Zenodo record {zenodo_id}...")
    meta = requests.get(
        f"https://zenodo.org/api/records/{zenodo_id}", timeout=30,
    ).json()
    if "files" not in meta:
        raise RuntimeError(f"Zenodo record {zenodo_id} has no files (response: {meta!r})")
    proof_py_url = next(
        (f["links"]["self"] for f in meta["files"] if f["key"] == "proof.py"),
        None,
    )
    if not proof_py_url:
        raise FileNotFoundError(f"No proof.py in Zenodo record {zenodo_id}")
    print(f"Downloading {proof_py_url}...")
    proof_py_text = requests.get(proof_py_url, timeout=60).text
    proof_engine_root = os.path.expanduser("~/proof-engine/proof-engine/skills/proof-engine")
    if not os.path.isdir(proof_engine_root):
        raise RuntimeError(
            f"proof-engine clone missing at {proof_engine_root}. "
            "postBuild should have cloned it — this image may be broken."
        )
    os.environ["PROOF_ENGINE_ROOT"] = proof_engine_root
    if proof_engine_root not in sys.path:
        sys.path.insert(0, proof_engine_root)
    print(f"Executing proof for DOI {doi}...\n")
    exec(compile(proof_py_text, f"proof_{zenodo_id}.py", "exec"), {"__name__": "__main__"})


widget = DOIReader(doi=_preloaded_doi())

def _on_trigger(change):
    output.clear_output()
    with output:
        try:
            run_proof((widget.doi or "").strip())
        except Exception as e:
            print(f"ERROR: {type(e).__name__}: {e}")
            raise

widget.observe(_on_trigger, names="trigger")
display(widget, output)

---

*Launcher repo: [yaniv-golan/proof-engine-binder](https://github.com/yaniv-golan/proof-engine-binder). See [yaniv-golan/proof-engine](https://github.com/yaniv-golan/proof-engine) for the proofs and skill.*